In [1]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, WhisperTokenizer,pipeline, WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset
import pandas as pd
import numpy as np

In [2]:
df_audio = pd.read_parquet('./data/parquets/testing_trained.parquet.gzip')
df_audio['snr_25_testing_trained']
df_audio['audio_SNR_25_path']
df_audio['SNR25_models_testing'] = df_audio['audio_SNR_25_path'].str.replace('\\', '/')

In [4]:
#training_evaluation_results_df = pd.DataFrame()
#training_evaluation_results_df['transcription'] = df_audio['ref_orig']

In [3]:
training_evaluation_results_df=pd.read_parquet('./data/parquets/training_evaluation_results.parquet.gzip')

In [4]:
torch.cuda.empty_cache()

In [5]:
df_audio

,audioname,dataset,ref_orig,sampling_rate,audiopath_bigos,audiopath_local,noise_path,noise_class,normalised_audio_path,normalised_noise_path,...,audio_SNR_0.1_path,audio_SNR_-1_path,audio_SNR_-3_path,audio_SNR_-10_path,snr_25_testing_trained,snr_10_testing_trained,snr_50_testing_trained,snr_100_testing_trained,snr_5_testing_trained,SNR25_models_testing
0,fair-mls-20-train-0009-04739,fair-mls-20,tam nocne włóczęgi wołano z dachów jeżeli nie ...,16000,fair-mls-20-train-0009-04739.wav,C:\Users\Eryk\.cache\huggingface\datasets\down...,.\data\UrbanSound8K\audio\fold9\79089-0-0-106.wav,air_conditioner,./data/mixed_recordings/normalised_recordings/...,./data/mixed_recordings/normalised_recordings/...,...,./data/mixed_recordings/SNR_0.1\fair-mls-20-tr...,./data/mixed_recordings/SNR_-1\fair-mls-20-tra...,./data/mixed_recordings/SNR_-3\fair-mls-20-tra...,./data/mixed_recordings/SNR_-10\fair-mls-20-tr...,./data/mixed_recordings/SNR_25_testing_new_mod...,./data/mixed_recordings/SNR_10_testing_new_mod...,./data/mixed_recordings/SNR_50_testing_new_mod...,./data/mixed_recordings/SNR_100_testing_new_mo...,./data/mixed_recordings/SNR_5_testing_new_mode...,./data/mixed_recordings/SNR_25/fair-mls-20-tra...
1,pjatk-clarin_studio-15-train-0488-00003,pjatk-clarin_studio-15,w pracy studenci chcieliby przede wszystk...,16000,pjatk-clarin_studio-15-train-0488-00003.wav,C:\Users\Eryk\.cache\huggingface\datasets\down...,.\data\UrbanSound8K\audio\fold10\167750-4-1-0.wav,drilling,./data/mixed_recordings/normalised_recordings/...,./data/mixed_recordings/normalised_recordings/...,...,./data/mixed_recordings/SNR_0.1\pjatk-clarin_s...,./data/mixed_recordings/SNR_-1\pjatk-clarin_st...,./data/mixed_recordings/SNR_-3\pjatk-clarin_st...,./data/mixed_recordings/SNR_-10\pjatk-clarin_s...,./data/mixed_recordings/SNR_25_testing_new_mod...,./data/mixed_recordings/SNR_10_testing_new_mod...,./data/mixed_recordings/SNR_50_testing_new_mod...,./data/mixed_recordings/SNR_100_testing_new_mo...,./data/mixed_recordings/SNR_5_testing_new_mode...,./data/mixed_recordings/SNR_25/pjatk-clarin_st...
2,fair-mls-20-train-0009-05501,fair-mls-20,co to znaczy sam siebie zapytywał faraon czy g...,16000,fair-mls-20-train-0009-05501.wav,C:\Users\Eryk\.cache\huggingface\datasets\down...,.\data\UrbanSound8K\audio\fold3\165039-7-5-0.wav,jackhammer,./data/mixed_recordings/normalised_recordings/...,./data/mixed_recordings/normalised_recordings/...,...,./data/mixed_recordings/SNR_0.1\fair-mls-20-tr...,./data/mixed_recordings/SNR_-1\fair-mls-20-tra...,./data/mixed_recordings/SNR_-3\fair-mls-20-tra...,./data/mixed_recordings/SNR_-10\fair-mls-20-tr...,./data/mixed_recordings/SNR_25_testing_new_mod...,./data/mixed_recordings/SNR_10_testing_new_mod...,./data/mixed_recordings/SNR_50_testing_new_mod...,./data/mixed_recordings/SNR_100_testing_new_mo...,./data/mixed_recordings/SNR_5_testing_new_mode...,./data/mixed_recordings/SNR_25/fair-mls-20-tra...
3,fair-mls-20-train-0021-01519,fair-mls-20,tylko na piaszczystem wybrzeżu lub na łąkach b...,16000,fair-mls-20-train-0021-01519.wav,C:\Users\Eryk\.cache\huggingface\datasets\down...,.\data\UrbanSound8K\audio\fold1\57320-0-0-22.wav,air_conditioner,./data/mixed_recordings/normalised_recordings/...,./data/mixed_recordings/normalised_recordings/...,...,./data/mixed_recordings/SNR_0.1\fair-mls-20-tr...,./data/mixed_recordings/SNR_-1\fair-mls-20-tra...,./data/mixed_recordings/SNR_-3\fair-mls-20-tra...,./data/mixed_recordings/SNR_-10\fair-mls-20-tr...,./data/mixed_recordings/SNR_25_testing_new_mod...,./data/mixed_recordings/SNR_10_testing_new_mod...,./data/mixed_recordings/SNR_50_testing_new_mod...,./data/mixed_recordings/SNR_100_testing_new_mo...,./data/mixed_recordings/SNR_5_testing_new_mode...,./data/mixed_recordings/SNR_25/fair-mls-20-tra...
4,pjatk-clarin_studio-15-train-0335-00001,pjatk-clarin_studio-15,zaokrągla uziemienie księdzu liźnięcie rol...,16000,pjatk-clarin_studio-15-train-0335-00001.wav,C:\Users\Eryk\.cache\huggingface\datasets\down...,.\data\Urban

In [6]:
training_evaluation_results_df.columns

Index(['transcription', 'SNR_25_model_no_train', 'SNR_25_model2_5k',
       'SNR_25_model3_5k', 'SNR_25_model10k', 'SNR_25_model15k',
       'SNR_50_model2_5k', 'SNR_50_model3_5k', 'SNR_50_model10k',
       'SNR_50_model15k', 'SNR_50_model35k', 'SNR_10_model2_5k',
       'SNR_10_model3_5k', 'SNR_10_model10k', 'SNR_10_model15k',
       'SNR_10_model35k', 'SNR_25_whisper_medium', 'SNR_5_whisper_medium',
       'SNR_5_model2_5k', 'SNR_5_model3_5k', 'SNR_5_model10k',
       'SNR_5_model15k', 'SNR_5_model35k', 'SNR_5_model80k', 'SNR_25_model35k',
       'SNR_25_model80k', 'SNR_10_model80k', 'SNR_100_model2_5k',
       'SNR_100_model3_5k', 'SNR_100_model10k', 'SNR_100_model15k',
       'SNR_100_model35k', 'SNR_100_model80k', 'SNR_50_model80k',
       'SNR_50_whisper_medium', 'SNR_10_whisper_medium'],
      dtype='object')

In [7]:
#models= [ "eryk7381/whisper-med-pol-car-2500", "eryk7381/whisper-med-pol-car-3500", "eryk7381/whisper-med-pol-car-10000" ,"eryk7381/whisper-med-pol-car-15000", "eryk7381/whisper-med-pol-car-35000","eryk7381/whisper-med-pol-car-80000"]
#columns = ["SNR_100_model2_5k", "SNR_100_model3_5k", "SNR_100_model10k", "SNR_100_model15k" , "SNR_100_model35k", "SNR_100_model80k"]

#models= ["eryk7381/whisper-med-pol-car-35000", "eryk7381/whisper-med-pol-car-80000"]
#columns = ["SNR_25_model35k", "SNR_25_model80k"]


models= ["openai/whisper-medium"]
columns = ["SNR_100_whisper_medium"]


columns_counter = 0


for model_id in models:

    temp_list = []

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
  
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True)
    model.to(device)

    processor = AutoProcessor.from_pretrained(model_id)

    pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype,
        device=device,
        chunk_length_s=30,
        generate_kwargs={"language": "pl", "task": "transcribe"}
    )

    
    for audio in df_audio['snr_100_testing_trained'].to_list():
        result = pipe(audio)
        temp_list.append(result["text"])

    training_evaluation_results_df[columns[columns_counter]] = temp_list

    columns_counter = columns_counter + 1

    torch.cuda.empty_cache()

c:\Users\Eryk\anaconda3\envs\Magisterka\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\Eryk\anaconda3\envs\Magisterka\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
c:\Users\Eryk\anaconda3\envs\Magisterka\Lib\site-packages\transformers\models\whisper\modeling_whisper.py:697: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.c

In [8]:
training_evaluation_results_df.columns

Index(['transcription', 'SNR_25_model_no_train', 'SNR_25_model2_5k',
       'SNR_25_model3_5k', 'SNR_25_model10k', 'SNR_25_model15k',
       'SNR_50_model2_5k', 'SNR_50_model3_5k', 'SNR_50_model10k',
       'SNR_50_model15k', 'SNR_50_model35k', 'SNR_10_model2_5k',
       'SNR_10_model3_5k', 'SNR_10_model10k', 'SNR_10_model15k',
       'SNR_10_model35k', 'SNR_25_whisper_medium', 'SNR_5_whisper_medium',
       'SNR_5_model2_5k', 'SNR_5_model3_5k', 'SNR_5_model10k',
       'SNR_5_model15k', 'SNR_5_model35k', 'SNR_5_model80k', 'SNR_25_model35k',
       'SNR_25_model80k', 'SNR_10_model80k', 'SNR_100_model2_5k',
       'SNR_100_model3_5k', 'SNR_100_model10k', 'SNR_100_model15k',
       'SNR_100_model35k', 'SNR_100_model80k', 'SNR_50_model80k',
       'SNR_50_whisper_medium', 'SNR_10_whisper_medium',
       'SNR_100_whisper_medium'],
      dtype='object')

In [9]:
training_evaluation_results_df.to_parquet('./data/parquets/training_evaluation_results.parquet.gzip', compression = 'gzip')

In [ ]:
import os
os.system('shutdown /s /t 0')